## Notebook Name: Landbird Habitat Cleaned
**Medallion Layer: Silver**  

**Purpose:** Does Data Validation, ensuring that the values of attributes are within the appropiate ranges

**Author:** Matthew Kristanto  

**Date Created:** 6/03/26  

**Last Modified:**  

**Source Table:** [bronze_landbird_habitat_injest](https://adb-7405618007730451.11.azuredatabricks.net/editor/notebooks/3335316465260836?o=7405618007730451) 


**Notes:**
- Ensures that Habitat_Num Column only has values 1 or 2
- Ensures that Is_Forested Column only has values of True or False
- Ensures that all of the Detection Class is Title Case


In [0]:
### Retrieve the Bronze Table

df_bronze = spark.read.table("bronze.bronze_landbird_habitat")

display(df_bronze.limit(5))

In [0]:
df_silver = df_bronze

### Retrieve the number of invalid habitat num values
def get_invalid_habitat_num_values(df_silver):
    df_invalid_habitat_num_values = df_silver.filter(~df_silver["Habitat_num"].isin(1,2))

    return df_invalid_habitat_num_values.count()

get_invalid_habitat_num_values(df_silver)

In [0]:
### Drop the invalid habitat num values
df_silver = df_silver.filter(df_silver["Habitat_num"].isin([1, 2]))


get_invalid_habitat_num_values(df_silver)

In [0]:
### Retrieve the values from Is_Forested Column that are not True or False

def get_invalid_is_forested_values():
  df_invalid_is_forested_values = df_silver.filter(~df_silver["Is_forested"].isin("TRUE", "FALSE"))

  display(df_invalid_is_forested_values)

  return df_invalid_is_forested_values.count()

get_invalid_is_forested_values()

In [0]:
### Drop the invalid values from Is_Forested
df_silver = df_silver.filter(df_silver["Is_forested"].isin("TRUE", "FALSE"))

get_invalid_is_forested_values()

In [0]:
### Retrieve all of the records that were not in Title Case for Detection_class
from pyspark.sql.functions import initcap, col

def get_invalid_detection_class_values(df_silver):
    df_invalid_detection_class_values = df_silver.filter(col("Detection_class") != initcap(col("Detection_class")))

    display(df_invalid_detection_class_values)

    return df_invalid_detection_class_values.count()


get_invalid_detection_class_values(df_silver)


In [0]:
### Convert all of the Data that is not Title Case for Detection Class into Title Case
from pyspark.sql.functions import initcap, col

def detection_class_to_title_case(df_silver):
    df_silver = df_silver.withColumn("Detection_class", initcap(col("Detection_class")))

    return df_silver


df_silver = detection_class_to_title_case(df_silver)

get_invalid_detection_class_values(df_silver)

In [0]:
### Create Silver Schema if not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
### Save the Silver Table to the Catalog
(df_silver.write
    .format("delta")  
    .mode("overwrite")  
    .saveAsTable("silver.silver_landbird_habitat_cleaned"))

In [0]:
%sql
SELECT * FROM silver.silver_landbird_habitat_cleaned